In [10]:
import sys
import pandas as pd
from sqlalchemy import text
from prime.config import get_engine

engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()

# Теперь q возвращает DataFrame, и .iloc сработает идеально
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]
WINDOW = ("2026-01-01", "2026-07-01")
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

print("Мой сегмент:", CLIENT_FILTER)

# Загружаем 4 нужные таблицы
clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
      and p.status = 'success'
""", a=WINDOW[0], b=WINDOW[1])

tickets = q(f"""
    select ticket_id, client_id, category, created_at, resolved_at
    from prime.support_tickets
    where client_id in ({SEGMENT})
      and created_at >= :a and created_at < :b
""", a=WINDOW[0], b=WINDOW[1])

for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("tickets", tickets)]:
    print(f"{name:<10} {len(frame):>9} строк")

Мой сегмент: client_id in (
    select client_id
    from prime.support_tickets
    where category = 'шеринг'
      and created_at >= '2025-10-01 00:00+00'
      and created_at <  '2026-04-01 00:00+00'
)
clients         6704 строк
subs            4621 строк
pay            15854 строк
tickets         4701 строк
